# Treinamento do detector — UFPR, RodoSol e combinado

Três experimentos independentes, cada um iniciado dos mesmos pesos YOLO26n pré-treinados, com uma única classe `placa`. As divisões oficiais são preservadas, conforme a [estratégia aprovada](../docs/3_estrategia_inicial_treinamento.md).

**Este notebook foi entregue sem iniciar treinos.** Revise os parâmetros propostos e altere `EXECUTAR_TREINOS` para `True` quando quiser executar. As células de treino são separadas; use apenas a desejada ou execute as três em sequência. Resultados ficam em `runs/detectores/<RODADA>`. Nenhum arquivo auxiliar é criado antes da execução autorizada pelos controles abaixo.

A avaliação usa somente validação. Os testes continuam reservados. Não altere a configuração no meio de uma rodada; para outra configuração use uma nova `RODADA` e refaça os três experimentos.


### 1. Localizar o projeto e conferir o ambiente

Use o kernel `.venv` do projeto. Não há instalação automática de pacotes. A primeira execução de um treino poderá baixar `yolo26n.pt` para `runs/pesos`, caso ainda não exista. GPU identificada durante a preparação: RTX 5070 Laptop, 8 GB; o batch proposto não é garantia de caber em qualquer configuração.


In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from time import perf_counter
from importlib.metadata import version
import gc
import json
import csv
import hashlib
import platform
import re
import yaml
import torch
from IPython.display import display

raiz = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets_yolo").is_dir()), None)
if raiz is None:
    raise FileNotFoundError("Abra este notebook na raiz do projeto ou em src.")
raiz = raiz.resolve()
VERSOES = {"python": platform.python_version(), "torch": torch.__version__, "ultralytics": version("ultralytics")}
print(VERSOES)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(gpu.name, f"{gpu.total_memory / 1024**3:.1f} GiB")


{'python': '3.14.5', 'torch': '2.11.0+cu128', 'ultralytics': '8.4.165'}
CUDA disponível: True
NVIDIA GeForce RTX 5070 Laptop GPU 8.0 GiB


### 2. Quais parâmetros serão iguais nos três experimentos?

**Proposta inicial, ainda não medida:** 100 épocas, 640 pixels, batch 8, AdamW com taxa inicial 0,001 e semente 42. `workers=0` simplifica a execução no notebook no Windows. Se faltar memória, diminua o batch e reinicie uma nova rodada completa; não há ajuste silencioso de batch.

Usamos `patience=0` para desabilitar parada antecipada e comparar `last.pt` após o mesmo número de épocas. Isso é um critério proposto de comparação, não uma escolha automática do melhor modelo. O `best.pt` padrão também é salvo, mas é selecionado pela validação usada em cada treino (diferente entre experimentos); não o confundimos com uma seleção comum nas validações complementares.

Mesmas épocas não significam mesmo número de atualizações nem mesmo custo. Registramos épocas, batches, atualizações efetivas do otimizador e tempo. Determinismo e semente ajudam a reproduzir, mas não garantem resultados idênticos entre versões/hardware.


In [2]:
EXECUTAR_TREINOS = False
EXECUTAR_VALIDACOES = False
RODADA = "comparacao_01"
PESOS_INICIAIS = raiz / "runs" / "pesos" / "yolo26n.pt"
CHECKPOINT_AVALIACAO = "last.pt"

PARAMETROS = dict(
    epochs=100,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    patience=0,
    seed=42,
    deterministic=True,
    workers=0,
    device=0 if torch.cuda.is_available() else "cpu",
    amp=torch.cuda.is_available(),
    cache=False,
    save=True,
    plots=True,
    val=True,
    resume=False,
)
if not re.fullmatch(r"[A-Za-z0-9_-]+", RODADA):
    raise ValueError("Use um nome simples para RODADA, sem separadores de caminho.")
PASTA_RODADA = raiz / "runs" / "detectores" / RODADA
DATASETS = {
    "UFPR": raiz / "datasets_yolo" / "UFPR" / "dataset.yaml",
    "RodoSol": raiz / "datasets_yolo" / "RodoSol" / "dataset.yaml",
    "Combinado": raiz / "datasets_yolo" / "combinado.yaml",
}
ESPERADOS = {"UFPR": (1800, 900), "RodoSol": (8000, 4000), "Combinado": (9800, 4900)}
print(json.dumps(PARAMETROS, indent=2))
print("Resultados:", PASTA_RODADA)


{
  "epochs": 100,
  "imgsz": 640,
  "batch": 8,
  "optimizer": "AdamW",
  "lr0": 0.001,
  "lrf": 0.01,
  "weight_decay": 0.0005,
  "warmup_epochs": 3.0,
  "patience": 0,
  "seed": 42,
  "deterministic": true,
  "workers": 0,
  "device": 0,
  "amp": true,
  "cache": false,
  "save": true,
  "plots": true,
  "val": true,
  "resume": false
}
Resultados: C:\Users\igorm\Documents\programacao\modelo-placa\runs\detectores\comparacao_01


### 3. Os YAMLs apontam para os dados esperados?

Conferimos treino e validação, incluindo os pares de labels. Não carregamos os testes. Esta checagem não substitui a auditoria de geometria feita nos notebooks de preparação.


In [3]:
def listar_imagens(config, chave):
    entradas = config[chave] if isinstance(config[chave], list) else [config[chave]]
    arquivos = []
    for entrada in entradas:
        pasta = Path(config["path"]) / entrada
        if not pasta.is_dir():
            raise FileNotFoundError(f"Pasta ausente: {pasta}. Reexecute os notebooks de preparação após mover o projeto.")
        arquivos.extend(sorted(p.resolve() for p in pasta.rglob("*") if p.suffix.lower() in (".png", ".jpg", ".jpeg")))
    if len(arquivos) != len(set(arquivos)):
        raise ValueError("Imagens repetidas na lista de caminhos do YAML")
    return arquivos

CONFIGURACOES = {}
ARQUIVOS = {}
for nome, caminho in DATASETS.items():
    config = yaml.safe_load(caminho.read_text(encoding="utf-8"))
    if config.get("names") != {0: "placa"}:
        raise ValueError(f"Classe inesperada: {caminho}")
    CONFIGURACOES[nome] = config
    ARQUIVOS[nome] = {}
    for chave, esperado in zip(("train", "val"), ESPERADOS[nome]):
        imagens = listar_imagens(config, chave)
        if len(imagens) != esperado:
            raise ValueError(f"{nome}/{chave}: {len(imagens)}, esperado {esperado}")
        for imagem in imagens:
            partes = list(imagem.parts)
            indice = len(partes) - 1 - partes[::-1].index("images")
            partes[indice] = "labels"
            label = Path(*partes).with_suffix(".txt")
            if not label.is_file():
                raise FileNotFoundError(label)
        ARQUIVOS[nome][chave] = imagens
        print(f"{nome}/{chave}: {len(imagens)} pares encontrados")
    if set(ARQUIVOS[nome]["train"]) & set(ARQUIVOS[nome]["val"]):
        raise ValueError(f"Caminhos compartilhados entre treino/validação: {nome}")
for chave in ("train", "val"):
    assert set(ARQUIVOS["Combinado"][chave]) == set(ARQUIVOS["UFPR"][chave]) | set(ARQUIVOS["RodoSol"][chave])
print("Preparação conferida. Textos/veículos compartilhados são tratados na avaliação complementar.")


UFPR/train: 1800 pares encontrados
UFPR/val: 900 pares encontrados
RodoSol/train: 8000 pares encontrados
RodoSol/val: 4000 pares encontrados
Combinado/train: 9800 pares encontrados
Combinado/val: 4900 pares encontrados
Preparação conferida. Textos/veículos compartilhados são tratados na avaliação complementar.


### 4. Como garantir que cada treino começa dos mesmos pesos?

A função cria uma nova instância `YOLO` para cada experimento, sempre a partir de `PESOS_INICIAIS`. Nunca usa os pesos de um treino anterior. Uma pasta de treino existente causa interrupção para evitar sobrescrever resultados. Para repetir um experimento, use outra rodada.

O resumo de execução é salvo dentro da própria pasta do treino. Em caso de interrupção, os checkpoints existentes são preservados; este notebook não retoma automaticamente um treino parcial.


In [4]:
def treinar(nome):
    if nome not in DATASETS:
        raise ValueError(f"Experimento desconhecido: {nome}")
    if not EXECUTAR_TREINOS:
        print(f"{nome}: treino desativado. Revise os parâmetros e altere EXECUTAR_TREINOS para True.")
        return None
    from ultralytics import YOLO

    destino = PASTA_RODADA / nome
    if destino.exists():
        raise FileExistsError(f"Experimento já existe: {destino}. Use outra RODADA; não será sobrescrito.")
    PESOS_INICIAIS.parent.mkdir(parents=True, exist_ok=True)
    modelo = YOLO(str(PESOS_INICIAIS), task="detect")
    hash_pesos = hashlib.sha256(PESOS_INICIAIS.read_bytes()).hexdigest()
    contadores = {"batches": 0, "atualizacoes_otimizador": 0, "epocas": 0}
    handles = []
    def contar_atualizacao(optimizer, args, kwargs):
        contadores["atualizacoes_otimizador"] += 1
    def instalar_contador(trainer):
        handles.append(trainer.optimizer.register_step_post_hook(contar_atualizacao))
    def contar_batch(trainer):
        contadores["batches"] += 1
    def contar_epoca(trainer):
        contadores["epocas"] = trainer.epoch + 1
    modelo.add_callback("on_train_start", instalar_contador)
    modelo.add_callback("on_train_batch_end", contar_batch)
    modelo.add_callback("on_train_epoch_end", contar_epoca)
    inicio = perf_counter()
    inicio_utc = datetime.now(timezone.utc).isoformat()
    estado = "interrompido"
    try:
        modelo.train(data=str(DATASETS[nome]), project=str(PASTA_RODADA), name=nome,
                     exist_ok=False, **PARAMETROS)
        checkpoint = destino / "weights" / CHECKPOINT_AVALIACAO
        if not checkpoint.is_file():
            raise FileNotFoundError(checkpoint)
        estado = "concluido"
        print("Checkpoint para comparação:", checkpoint)
        return checkpoint
    finally:
        for handle in handles:
            handle.remove()
        if destino.is_dir():
            registro = dict(experimento=nome, estado=estado, inicio_utc=inicio_utc,
                            segundos=perf_counter()-inicio, **contadores,
                            parametros=PARAMETROS, versoes=VERSOES,
                            pesos_iniciais=str(PESOS_INICIAIS), sha256_pesos_iniciais=hash_pesos,
                            dataset=str(DATASETS[nome]), checkpoint_avaliacao=CHECKPOINT_AVALIACAO)
            (destino / "execucao.json").write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
        del modelo
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


### Treino — UFPR

1.800 imagens de treino; validação oficial UFPR com 900 imagens.


In [5]:
checkpoint_ufpr = treinar("UFPR")


UFPR: treino desativado. Revise os parâmetros e altere EXECUTAR_TREINOS para True.


### Treino — RodoSol

8.000 imagens de treino; validação oficial RodoSol com 4.000 imagens.


In [6]:
checkpoint_rodosol = treinar("RodoSol")


RodoSol: treino desativado. Revise os parâmetros e altere EXECUTAR_TREINOS para True.


### Treino — Combinado

9.800 imagens de treino, sem reamostragem; validação agregada com 4.900 imagens. A comparação por origem vem depois.


In [7]:
checkpoint_combinado = treinar("Combinado")


Combinado: treino desativado. Revise os parâmetros e altere EXECUTAR_TREINOS para True.


### 5. Como avaliar cada detector nas duas origens?

Após treinar, altere `EXECUTAR_VALIDACOES` para `True`. São 12 avaliações: três modelos × duas origens × validação completa/complementar. Sempre `split="val"`; os testes não são usados.

Na complementar UFPR, omitimos uma das duas cópias idênticas de `track0090`. Na RodoSol, omitimos as três imagens de HDE1F59, compartilhada com treino. As listas são criadas dentro da rodada, sem mover ou apagar imagens. Esses filtros não garantem ausência de todo vazamento.

Avalie principalmente os resultados complementares por origem; médias agregadas não definem automaticamente o vencedor. O acerto da placa inteira depende da etapa posterior de OCR e não é medido por este notebook.


In [8]:
def preparar_validacoes_complementares():
    pasta = PASTA_RODADA / "configuracoes_validacao"
    pasta.mkdir(parents=True, exist_ok=True)
    saida = {}
    for nome in ("UFPR", "RodoSol"):
        mantidas, excluidas = [], []
        for imagem in ARQUIVOS[nome]["val"]:
            relativo = imagem.relative_to(raiz / "datasets_yolo" / nome / "images" / "validation")
            if nome == "UFPR":
                excluir = relativo.as_posix() == "track0090/track0090[15].png"
                if excluir:
                    par = imagem.with_name("track0090[14].png")
                    if hashlib.sha256(imagem.read_bytes()).digest() != hashlib.sha256(par.read_bytes()).digest():
                        raise ValueError("O par conhecido deixou de ser idêntico; revisar o filtro UFPR")
            else:
                anotacao = raiz / "datasets_brutos" / "tbFcZE-RodoSol-ALPR" / "images" / relativo.with_suffix(".txt")
                linhas = anotacao.read_text(encoding="utf-8").splitlines()
                placas = [linha.partition(":")[2].strip().upper() for linha in linhas if linha.partition(":")[0].strip() == "plate"]
                if len(placas) != 1:
                    raise ValueError(f"Texto de placa ambíguo/ausente: {anotacao}")
                excluir = placas[0] == "HDE1F59"
            (excluidas if excluir else mantidas).append(imagem)
        esperado = 1 if nome == "UFPR" else 3
        if len(excluidas) != esperado:
            raise ValueError(f"Filtro {nome}: {len(excluidas)} exclusões; esperado {esperado}. Revisar dados.")
        lista = pasta / f"{nome}_val_complementar.txt"
        lista.write_text("\n".join(p.as_posix() for p in mantidas) + "\n", encoding="utf-8")
        config = {**CONFIGURACOES[nome], "val": lista.as_posix()}
        config.pop("test", None)
        arquivo = pasta / f"{nome}_val_complementar.yaml"
        arquivo.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False), encoding="utf-8")
        saida[nome] = arquivo
        print(nome, len(mantidas), "imagens na validação complementar")
    return saida

resultados_validacao = []
if not EXECUTAR_VALIDACOES:
    print("Validações desativadas. Execute após concluir os três treinos.")
else:
    from ultralytics import YOLO
    checkpoints = {nome: PASTA_RODADA / nome / "weights" / CHECKPOINT_AVALIACAO for nome in DATASETS}
    ausentes = [str(p) for p in checkpoints.values() if not p.is_file()]
    if ausentes:
        raise FileNotFoundError("Checkpoints ausentes: " + "; ".join(ausentes))
    hashes = set()
    for nome in DATASETS:
        registro = json.loads((PASTA_RODADA / nome / "execucao.json").read_text(encoding="utf-8"))
        if registro["estado"] != "concluido" or registro["epocas"] != PARAMETROS["epochs"]:
            raise ValueError(f"Treino incompleto: {nome}; não comparar checkpoints parciais.")
        if registro["parametros"] != PARAMETROS:
            raise ValueError(f"Parâmetros da rodada diferem dos atuais: {nome}")
        hashes.add(registro["sha256_pesos_iniciais"])
    if len(hashes) != 1:
        raise ValueError("Os treinos partiram de pesos diferentes")
    complementares = preparar_validacoes_complementares()
    for experimento, checkpoint in checkpoints.items():
        modelo = YOLO(str(checkpoint))
        for origem in ("UFPR", "RodoSol"):
            for protocolo, config in (("oficial", DATASETS[origem]), ("complementar", complementares[origem])):
                metricas = modelo.val(data=str(config), split="val", imgsz=PARAMETROS["imgsz"],
                    batch=PARAMETROS["batch"], device=PARAMETROS["device"], workers=0,
                    project=str(PASTA_RODADA / "validacoes"),
                    name=f"{experimento}_{origem}_{protocolo}", exist_ok=False,
                    plots=True, save_json=False, seed=PARAMETROS["seed"], deterministic=True)
                resultados_validacao.append(dict(treino=experimento, origem=origem, protocolo=protocolo,
                    precisao=float(metricas.box.mp), recall=float(metricas.box.mr),
                    map50=float(metricas.box.map50), map50_95=float(metricas.box.map)))
        del modelo
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    (PASTA_RODADA / "comparacao_validacao.json").write_text(
        json.dumps(resultados_validacao, ensure_ascii=False, indent=2), encoding="utf-8")
    display(resultados_validacao)


Validações desativadas. Execute após concluir os três treinos.


### 6. Como comparar qualidade e custo?

`results.csv` e os gráficos de cada treino são gerados pela Ultralytics. `execucao.json` registra parâmetros e custo; `comparacao_validacao.json` guarda os resultados separados por origem/protocolo. Os resultados do experimento permanecem dentro de `runs`, criado somente ao executar os treinos/validações. A Ultralytics também pode criar caches de labels junto aos datasets e manter pesos auxiliares na pasta configurada pela biblioteca para a checagem AMP.

Observe especialmente recall de placas pequenas e diferenças entre origens. Uma queda da loss não garante melhoria do sistema. Cenas com placas visíveis não anotadas podem distorcer a precisão; inspeção de erros e avaliação por tipo/layout são próximos passos antes da avaliação final no teste.


In [9]:
resumos = []
for nome in DATASETS:
    arquivo = PASTA_RODADA / nome / "execucao.json"
    if arquivo.is_file():
        r = json.loads(arquivo.read_text(encoding="utf-8"))
        resumos.append({k: r[k] for k in ("experimento", "estado", "epocas", "batches", "atualizacoes_otimizador", "segundos", "sha256_pesos_iniciais")})
if resumos:
    display(resumos)
    if len({r["sha256_pesos_iniciais"] for r in resumos}) != 1:
        raise ValueError("Os experimentos partiram de pesos diferentes; não são a mesma rodada comparável.")
else:
    print("Ainda não há treinos registrados nesta rodada.")


Ainda não há treinos registrados nesta rodada.
